# LLM Classification Finetuning — BERT-based Model
**Competition:** LLM Classification Finetuning (Kaggle)  
**Goal:** Predict winner_model_a / winner_model_b / tie given two LLM responses  
**Hardware:** Kaggle T4 x2  
**Strategy:** DeBERTa-v3-base fine-tuned on concatenated [prompt + response_a + response_b] with multi-GPU support, aggressive cleaning, and smart truncation.

## 1. Install & Imports

In [ ]:
%%capture
!pip install transformers==4.40.1 datasets accelerate sentencepiece protobuf -q

In [ ]:
import os, gc, re, math, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
    DataCollatorWithPadding,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
seed_everything()

# ── Device ───────────────────────────────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPUs available: {NUM_GPUS}  |  Device: {DEVICE}')

## 2. Configuration

In [ ]:
class CFG:
    # ── Paths ─────────────────────────────────────────────────────────────────
    DATA_DIR   = Path('/kaggle/input/competitions/llm-classification-finetuning')
    OUTPUT_DIR = Path('/kaggle/working')

    # ── Model ─────────────────────────────────────────────────────────────────
    # DeBERTa-v3-base is the best BERT-family model for classification tasks
    MODEL_NAME  = 'microsoft/deberta-v3-base'
    NUM_LABELS  = 3          # winner_model_a, winner_model_b, tie
    MAX_LEN     = 1024       # tokens — T4 handles this with AMP

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS          = 2
    BATCH_SIZE      = 4      # per GPU
    GRAD_ACCUM      = 4      # effective batch = 4 * 4 * 2 GPUs = 32
    LR              = 2e-5
    WEIGHT_DECAY    = 0.01
    WARMUP_RATIO    = 0.1
    MAX_GRAD_NORM   = 1.0
    N_FOLDS         = 3
    TRAIN_FOLDS     = [0, 1, 2]   # train all folds; change to [0] for quick test

    # ── Augmentation ──────────────────────────────────────────────────────────
    # Swap A/B responses with label inversion (doubles data, handles position bias)
    SWAP_AUG = True

    # ── Misc ──────────────────────────────────────────────────────────────────
    FP16       = True
    NUM_WORKERS = 2

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config loaded.')

## 3. Load Data

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv')
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv')
sub_df   = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv')

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
print(f'\nTrain columns: {list(train_df.columns)}')
print(f'\nLabel distribution:')
print(train_df[['winner_model_a','winner_model_b','winner_tie']].sum())

## 4. Dataset Cleaning & Preprocessing

In [ ]:
# ── 4.1 Build unified label column ───────────────────────────────────────────
def extract_label(row):
    """Convert three binary winner columns into a single string label."""
    if row.get('winner_model_a', 0) == 1:
        return 'winner_model_a'
    elif row.get('winner_model_b', 0) == 1:
        return 'winner_model_b'
    else:
        return 'winner_tie'

# Handle both possible column naming conventions
if 'winner_model_a' in train_df.columns:
    train_df['label'] = train_df.apply(extract_label, axis=1)
elif 'winner_tie' not in train_df.columns and 'winner_model_a' not in train_df.columns:
    # Some versions use winner_model_[a/b/tie] as separate columns
    # Re-check actual format
    label_cols = [c for c in train_df.columns if 'winner' in c]
    print('Found label columns:', label_cols)
    train_df['label'] = train_df[label_cols].idxmax(axis=1)

LABEL2ID = {'winner_model_a': 0, 'winner_model_b': 1, 'winner_tie': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
train_df['label_id'] = train_df['label'].map(LABEL2ID)

print('Label distribution:\n', train_df['label'].value_counts())
train_df.head(3)

In [ ]:
# ── 4.2 Text Cleaning ─────────────────────────────────────────────────────────
def clean_text(text):
    """Robust text cleaner for LLM response data."""
    if not isinstance(text, str):
        return ''
    # Remove null bytes and control characters (except newline/tab)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    # Collapse excessive whitespace / blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    # Replace common unicode artifacts
    text = text.replace('\u200b', '').replace('\ufeff', '')
    return text

for col in ['prompt', 'response_a', 'response_b']:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col]  = test_df[col].apply(clean_text)

# ── 4.3 Drop rows with empty critical fields ───────────────────────────────────
before = len(train_df)
train_df = train_df[
    (train_df['prompt'].str.len() > 0) &
    ((train_df['response_a'].str.len() > 0) | (train_df['response_b'].str.len() > 0))
].reset_index(drop=True)
print(f'Dropped {before - len(train_df)} rows with empty fields. Remaining: {len(train_df)}')

# ── 4.4 Fill NaN responses ────────────────────────────────────────────────────
train_df['response_a'] = train_df['response_a'].fillna('[NO RESPONSE]')
train_df['response_b'] = train_df['response_b'].fillna('[NO RESPONSE]')
test_df['response_a']  = test_df['response_a'].fillna('[NO RESPONSE]')
test_df['response_b']  = test_df['response_b'].fillna('[NO RESPONSE]')

print('Cleaning complete.')

In [ ]:
# ── 4.5 EDA: text length distribution ─────────────────────────────────────────
for col in ['prompt', 'response_a', 'response_b']:
    lens = train_df[col].str.split().str.len()
    print(f'{col:12s} | mean={lens.mean():.0f}  median={lens.median():.0f}  p95={lens.quantile(0.95):.0f}  max={lens.max()}')

## 5. Tokenizer & Smart Truncation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

# Special separator tokens for structured input
SEP = tokenizer.sep_token or '[SEP]'

def build_input_text(prompt, response_a, response_b, max_len_chars=4096):
    """
    Build a structured input string with smart truncation.
    Format:  [PROMPT] <prompt> [A] <response_a> [B] <response_b>
    Truncates response texts proportionally to fit within char limit,
    always preserving the beginning (most informative for quality judgment).
    """
    header_a = '[PROMPT] '
    header_b = ' [RESPONSE A] '
    header_c = ' [RESPONSE B] '
    
    # Reserve chars for headers
    overhead = len(header_a) + len(header_b) + len(header_c)
    budget   = max_len_chars - overhead

    # Proportional budget: 20% prompt, 40% A, 40% B
    p_budget = int(budget * 0.20)
    r_budget = int(budget * 0.40)

    p_text  = prompt[:p_budget]
    ra_text = response_a[:r_budget]
    rb_text = response_b[:r_budget]

    return f"{header_a}{p_text}{header_b}{ra_text}{header_c}{rb_text}"

# Quick sanity check
sample = build_input_text(
    train_df['prompt'].iloc[0],
    train_df['response_a'].iloc[0],
    train_df['response_b'].iloc[0]
)
print(f'Sample input length (chars): {len(sample)}')
tokens = tokenizer(sample, truncation=True, max_length=CFG.MAX_LEN)
print(f'Sample input length (tokens): {len(tokens["input_ids"])}')

## 6. Dataset & DataLoader

In [ ]:
class LLMDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.is_train  = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = build_input_text(row['prompt'], row['response_a'], row['response_b'])
        
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,         # collator handles padding
            return_tensors=None,
        )
        item = {
            'input_ids':      torch.tensor(enc['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = torch.tensor(enc['token_type_ids'], dtype=torch.long)
        
        if self.is_train:
            item['labels'] = torch.tensor(row['label_id'], dtype=torch.long)
        return item


def get_loaders(train_df, val_df, tokenizer):
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='longest')
    
    train_ds = LLMDataset(train_df, tokenizer, CFG.MAX_LEN, is_train=True)
    val_ds   = LLMDataset(val_df,   tokenizer, CFG.MAX_LEN, is_train=True)
    
    train_loader = DataLoader(
        train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=False,
        collate_fn=collator
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        collate_fn=collator
    )
    return train_loader, val_loader

print('Dataset classes defined.')

## 7. Swap Augmentation

In [ ]:
def swap_augment(df):
    """
    Swap response_a <-> response_b and invert the label.
    - winner_model_a  -> winner_model_b
    - winner_model_b  -> winner_model_a
    - winner_tie -> winner_tie  (symmetric)
    This doubles training data and removes position bias.
    """
    swapped = df.copy()
    swapped['response_a'] = df['response_b']
    swapped['response_b'] = df['response_a']

    def invert_label(lbl):
        if lbl == 'winner_model_a': return 'winner_model_b'
        if lbl == 'winner_model_b': return 'winner_model_a'
        return lbl  # tie stays tie

    swapped['label']    = swapped['label'].apply(invert_label)
    swapped['label_id'] = swapped['label'].map(LABEL2ID)

    augmented = pd.concat([df, swapped], ignore_index=True).sample(frac=1, random_state=SEED)
    return augmented.reset_index(drop=True)

print('Swap augmentation function ready.')

## 8. Model

In [ ]:
def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.MODEL_NAME,
        num_labels=CFG.NUM_LABELS,
        ignore_mismatched_sizes=True,
    )
    # Multi-GPU DataParallel (T4 x2)
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
    model = model.to(DEVICE)
    return model

print('Model builder ready.')

## 9. Training & Validation Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, grad_accum):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with autocast(enabled=CFG.FP16):
            outputs = model(**batch)
            loss    = outputs.loss
            if NUM_GPUS > 1:
                loss = loss.mean()   # DataParallel returns per-GPU losses
            loss = loss / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum

        if (step + 1) % 50 == 0:
            print(f'  Step [{step+1}/{len(loader)}]  loss={total_loss/(step+1):.4f}')

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_logits = []
    all_labels = []

    for batch in loader:
        labels = batch.pop('labels').to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}

        with autocast(enabled=CFG.FP16):
            outputs = model(**batch)

        all_logits.append(outputs.logits.cpu().float())
        all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0).numpy()

    probs  = torch.softmax(all_logits, dim=-1).numpy()
    preds  = np.argmax(probs, axis=1)

    # One-hot encode for log_loss
    n      = len(all_labels)
    onehot = np.zeros((n, CFG.NUM_LABELS))
    onehot[np.arange(n), all_labels] = 1

    ll  = log_loss(onehot, probs)
    acc = accuracy_score(all_labels, preds)
    return ll, acc, probs


print('Training functions defined.')

## 10. K-Fold Cross Validation Training

In [ ]:
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)

oof_probs   = np.zeros((len(train_df), CFG.NUM_LABELS))
test_probs  = np.zeros((len(test_df),  CFG.NUM_LABELS))

# ── Test dataset (static, shared across folds) ────────────────────────────────
collator   = DataCollatorWithPadding(tokenizer=tokenizer, padding='longest')
test_ds    = LLMDataset(test_df, tokenizer, CFG.MAX_LEN, is_train=False)
test_loader = DataLoader(
    test_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True, collate_fn=collator
)

fold_scores = []  # (log_loss, accuracy) per fold

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label_id'])):
    if fold not in CFG.TRAIN_FOLDS:
        continue

    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{CFG.N_FOLDS}')
    print(f'{"="*60}')

    tr_df  = train_df.iloc[train_idx].copy()
    val_df = train_df.iloc[val_idx].copy()

    # Apply swap augmentation only on training split
    if CFG.SWAP_AUG:
        tr_df = swap_augment(tr_df)
        print(f'  Train size after aug: {len(tr_df)} | Val size: {len(val_df)}')
    else:
        print(f'  Train size: {len(tr_df)} | Val size: {len(val_df)}')

    train_loader, val_loader = get_loaders(tr_df, val_df, tokenizer)

    # ── Model, optimizer, scheduler ───────────────────────────────────────────
    model = build_model()

    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    param_groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': CFG.WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(param_groups, lr=CFG.LR)

    total_steps   = (len(train_loader) // CFG.GRAD_ACCUM) * CFG.EPOCHS
    warmup_steps  = int(total_steps * CFG.WARMUP_RATIO)
    scheduler     = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler        = GradScaler(enabled=CFG.FP16)

    best_ll       = float('inf')
    best_ckpt     = CFG.OUTPUT_DIR / f'fold{fold}_best.pt'

    for epoch in range(CFG.EPOCHS):
        print(f'\n  Epoch {epoch+1}/{CFG.EPOCHS}')
        tr_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, CFG.GRAD_ACCUM)
        val_ll, val_acc, val_probs = validate(model, val_loader)

        print(f'  >>> Train loss={tr_loss:.4f}  Val LogLoss={val_ll:.4f}  Val Acc={val_acc:.4f}')

        if val_ll < best_ll:
            best_ll = val_ll
            # Save only model weights (strip DataParallel wrapper if needed)
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state, best_ckpt)
            print(f'  ✓ Saved best checkpoint (LogLoss={best_ll:.4f})')

    # ── Load best checkpoint for OOF and test inference ───────────────────────
    model_core = AutoModelForSequenceClassification.from_pretrained(
        CFG.MODEL_NAME, num_labels=CFG.NUM_LABELS, ignore_mismatched_sizes=True
    )
    model_core.load_state_dict(torch.load(best_ckpt, map_location='cpu'))
    if NUM_GPUS > 1:
        model_core = nn.DataParallel(model_core)
    model_core = model_core.to(DEVICE)

    # OOF predictions (on un-augmented val split)
    val_ll, val_acc, val_probs = validate(model_core, val_loader)
    oof_probs[val_idx] = val_probs
    fold_scores.append((val_ll, val_acc))
    print(f'\n  Fold {fold+1} Best — LogLoss={val_ll:.4f}  Accuracy={val_acc:.4f}')

    # Test inference
    model_core.eval()
    fold_test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=CFG.FP16):
                out = model_core(**batch)
            fold_test_probs.append(torch.softmax(out.logits, dim=-1).cpu().float().numpy())
    test_probs += np.concatenate(fold_test_probs, axis=0) / len(CFG.TRAIN_FOLDS)

    del model, model_core; gc.collect(); torch.cuda.empty_cache()

print('\n' + '='*60)
print('ALL FOLDS COMPLETE')
print('='*60)

## 11. OOF Evaluation

In [ ]:
# Per-fold scores
print('Per-fold scores:')
for i, (ll, acc) in enumerate(fold_scores):
    print(f'  Fold {i+1} — LogLoss: {ll:.5f}  Accuracy: {acc:.5f}')

# Overall OOF score
n = len(train_df)
onehot = np.zeros((n, CFG.NUM_LABELS))
onehot[np.arange(n), train_df['label_id'].values] = 1

oof_ll  = log_loss(onehot, oof_probs)
oof_acc = accuracy_score(train_df['label_id'].values, np.argmax(oof_probs, axis=1))

print(f'\n★ OOF LogLoss : {oof_ll:.5f}')
print(f'★ OOF Accuracy: {oof_acc:.5f}')

In [ ]:
# Confusion matrix on OOF
from sklearn.metrics import confusion_matrix, classification_report

oof_preds = np.argmax(oof_probs, axis=1)
labels_true = train_df['label_id'].values
label_names = ['winner_model_a', 'winner_model_b', 'winner_tie']

cm = confusion_matrix(labels_true, oof_preds)
print('Confusion Matrix:')
print(pd.DataFrame(cm, index=label_names, columns=label_names))
print('\nClassification Report:')
print(classification_report(labels_true, oof_preds, target_names=label_names))

## 12. Generate Submission

In [ ]:
# Map probabilities -> submission format
# The submission has columns: id, winner_model_a, winner_model_b, winner_tie
sub_df['winner_model_a']   = test_probs[:, 0]
sub_df['winner_model_b']   = test_probs[:, 1]
sub_df['winner_tie'] = test_probs[:, 2]

sub_df.to_csv(CFG.OUTPUT_DIR / 'submission.csv', index=False)
print('Submission saved.')
print(sub_df.head())

## 13. Summary

In [ ]:
print('='*60)
print('TRAINING SUMMARY')
print('='*60)
print(f'Model          : {CFG.MODEL_NAME}')
print(f'Max token len  : {CFG.MAX_LEN}')
print(f'Epochs / Fold  : {CFG.EPOCHS}')
print(f'Folds trained  : {CFG.TRAIN_FOLDS}')
print(f'Swap Aug       : {CFG.SWAP_AUG}')
print(f'FP16           : {CFG.FP16}')
print(f'Grad Accum     : {CFG.GRAD_ACCUM}')
print()
print(f'★ OOF LogLoss  : {oof_ll:.5f}')
print(f'★ OOF Accuracy : {oof_acc:.5f}')
print('='*60)